# Optuna `optuna_nogate` — TCN/LSTM sem persistence gate (Colab)

Alternativa ao Kaggle caso o accelerator esteja indisponível.

**Setup:**
1. Gere o pacote localmente: `bash scripts/kaggle_package.sh`
2. Runtime → Change runtime type → **GPU (T4)**
3. Rode as células na ordem; na célula 3, faça upload de
   `dist/kaggle_optuna_<data>.zip` quando o prompt abrir
4. Autorize o acesso ao Google Drive na célula 4 — o progresso do estudo é
   sincronizado para `MyDrive/optuna_nogate_backup/` a cada 20 min, então uma
   desconexão não perde os trials

**Retomar depois de uma desconexão:** rode novamente e, na célula 3, escolha a
opção de restaurar do Drive (o sqlite é recolocado no lugar e o estudo
continua do último checkpoint).

Colab gratuito: T4 16 GB, sessões variáveis (~4–12 h). O `step_optuna` grava
cada trial no sqlite e já poda trials que estourarem a GPU.

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
%pip install -q "tensorflow[and-cuda]==2.21.0" "keras==3.12.1" "keras-tcn==3.5.6" "optuna==4.9.0" "PyWavelets==1.8.0" "pandas>=2.0" "scikit-learn>=1.3" "numpy>=1.26"
import tensorflow as tf
print("TF apos pin:", tf.__version__)

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

WORK = Path("/content")
REPO = WORK / "wind-speed-forecasting"
REPO.mkdir(parents=True, exist_ok=True)

print("Selecione dist/kaggle_optuna_<data>.zip gerado pelo scripts/kaggle_package.sh")
up = files.upload()
repo_zip = WORK / next(iter(up))
with zipfile.ZipFile(repo_zip) as z:
    z.extractall(REPO)

restore = input("Restaurar progresso do Drive? (caminho no Drive ou vazio p/ pular): ").strip()
if restore:
    from google.colab import drive
    drive.mount("/content/drive")
    progress = Path(restore)
    if progress.is_dir():
        target = REPO / "pipeline" / "tmp" / "optuna_nogate"
        target.mkdir(parents=True, exist_ok=True)
        import shutil
        for f in progress.glob("*/optuna.db"):
            shutil.copy2(f, target / f.parent.name / f.name)
        for f in progress.glob("*/*.json") + progress.glob("*/*.csv"):
            shutil.copy2(f, target / f.parent.name / f.name)
    else:
        with zipfile.ZipFile(progress) as z:
            z.extractall(REPO)
    print("progresso restaurado de", restore)

assert (REPO / "data" / "dataset.csv").is_file(), "dataset.csv ausente"
dbs = sorted(REPO.glob("pipeline/tmp/optuna_nogate/*/optuna.db"))
print("repo em:", REPO)
print("dbs:", [str(d.relative_to(REPO)) for d in dbs] or "nenhum")

In [ ]:
import shutil
import threading
import time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
BACKUP = Path("/content/drive/MyDrive/optuna_nogate_backup")
SRC = REPO / "pipeline" / "tmp" / "optuna_nogate"
BACKUP.mkdir(parents=True, exist_ok=True)

def sync_loop(interval=1200):
    while True:
        if SRC.is_dir():
            for db_dir in SRC.iterdir():
                if db_dir.is_dir():
                    shutil.copytree(db_dir, BACKUP / db_dir.name, dirs_exist_ok=True)
        time.sleep(interval)

threading.Thread(target=sync_loop, daemon=True).start()
print("backup periodico para", BACKUP, "a cada 20 min — mantenha esta celula viva")

In [ ]:
import os

os.chdir("/content/wind-speed-forecasting")
!python pipeline/step_optuna.py --config pipeline/pipeline.optuna_nogate.json

In [ ]:
import glob
import json
import shutil

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
for db in sorted(glob.glob("pipeline/tmp/optuna_nogate/*/optuna.db")):
    name = db.split("/")[-2]
    study = optuna.load_study(study_name="optuna_nogate", storage=f"sqlite:///{db}")
    done = [t for t in study.trials if t.state.name == "COMPLETE"]
    best = min((t.value for t in done), default=float("nan"))
    print(f"{name}: {len(study.trials)}/100 trials | melhor block_mse={best:.4f}")
    for f in glob.glob("pipeline/tmp/optuna_nogate/*/best_trial.json"):
        if name in f:
            with open(f) as handle:
                bt = json.load(handle)
            print(json.dumps(bt["best_params"], indent=2))

for db_dir in SRC.iterdir():
    if db_dir.is_dir():
        shutil.copytree(db_dir, BACKUP / db_dir.name, dirs_exist_ok=True)
shutil.make_archive("/content/optuna_nogate", "zip", SRC)
from google.colab import files as colab_files
colab_files.download("/content/optuna_nogate.zip")
print("optuna_nogate.zip tambem esta em MyDrive/optuna_nogate_backup/")